# `medsig` playground

This notebook illustrates the two counting models used in the paper. Run it from an editable installation (`python3 -m pip install -e .`).

The discovery statistics are $q_0=[\max(0,r)]^2$ and $q_0^*=[\max(0,r^*)]^2$; their significances are the corresponding square roots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.on import expected_significance_on, pvals_on
from src.on_off import asimov_Zs_onoff, pvals_onoff_profile_sum

## Known background

Here $N\sim\operatorname{Pois}(s+b)$. The exact reference is the inclusive Poisson tail $P(N\geq n\mid s_0+b)$.

In [ ]:
s0, b = 0.0, 1.0
n_values = np.arange(0, 8)
pvalues = pvals_on(s0, b, n_values)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(n_values, pvalues["p_exact"], "x", label="Exact")
ax.semilogy(n_values, pvalues["p_r"], "o", label=r"$1-\Phi(\sqrt{q_0})$")
ax.semilogy(n_values, pvalues["p_rstar"], "^", label=r"$1-\Phi(\sqrt{q_0^*})$")
ax.set(xlabel=r"Observed count $n$", ylabel="p-value")
ax.grid(True, which="both", linestyle="--", alpha=0.35)
ax.legend(frameon=False);

The next cell compares the two Asimov approximations with the median from repeated Poisson observations.

In [ ]:
b_values = np.logspace(-1, 2, 40)
summary = expected_significance_on(2.0, b_values, n_outer=500, seed=12345)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(b_values, summary["Z_A_r"], label=r"Asimov $q_0$")
ax.plot(b_values, summary["Z_A_rstar"], "--", label=r"Asimov $q_0^*")
ax.plot(b_values, summary["Z_mc_median"], "o", label="MC median")
ax.set(xscale="log", xlabel=r"Background $b$", ylabel=r"$\operatorname{med}[Z\mid s]$")
ax.grid(True, which="both", linestyle="--", alpha=0.35)
ax.legend(frameon=False);

## On/off measurement

The uncertain-background model is $N\sim\operatorname{Pois}(s+b)$ and $M\sim\operatorname{Pois}(\tau b)$. `profile_sum` fixes the background to its profile estimate and deterministically sums the inclusive joint-Poisson tail.

In [ ]:
observed = pvals_onoff_profile_sum(s=0.0, n=5, m=1, tau=1.0)
asimov = asimov_Zs_onoff(s_true=2.0, b=1.0, tau=1.0)

print("Observed p-values:", observed)
print("Asimov significances:", asimov)